# Sessão 1 — Conectar

In [8]:
import pika

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

In [9]:
print(type(channel))

<class 'pika.adapters.blocking_connection.BlockingChannel'>


# Sessão 2 — Criar a fila e publicar uma mensagem

In [10]:
channel.queue_declare(queue='lab_churn', durable=True)

channel.basic_publish(exchange='', 
                      routing_key='lab_churn', 
                      body='{"customer_id": 1, "tenure": 12, "contract": "Month-to-month"}'
                      )

print("Mensagem publicada!")

Mensagem publicada!


# Sessão 3 — Consumir a mensagem

In [11]:
def processar_mensagem(ch, method, properties, body):
    import json
    dados = json.loads(body)
    print("Mensagem recebida:", dados)
    print(f'customer_id: {dados["customer_id"]}')
    print(f'tenure: {dados["tenure"]}')
    print(f'contract: {dados["contract"]}')

    # ACK — avisa o RabbitMQ que processou com sucesso
    # Só depois disso a mensagem some da fila
    ch.basic_ack(delivery_tag=method.delivery_tag)
    print("Mensagem ACKed!")
channel.basic_consume(queue='lab_churn', on_message_callback=processar_mensagem)
print("Aguardando mensagens...")

Aguardando mensagens...


In [12]:
connection.process_data_events(time_limit=5)  # Processa mensagens por 5 segundos

Mensagem recebida: {'customer_id': 1, 'tenure': 12, 'contract': 'Month-to-month'}
customer_id: 1
tenure: 12
contract: Month-to-month
Mensagem ACKed!
Mensagem recebida: {'customer_id': 1, 'tenure': 12, 'contract': 'Month-to-month'}
customer_id: 1
tenure: 12
contract: Month-to-month
Mensagem ACKed!


### Sessão 4 - Cenário com Falha

In [1]:
import pika

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()

In [2]:
channel.queue_declare(queue='lab_churn', durable=True)
channel.basic_publish(exchange='', 
                      routing_key='lab_churn',
                      body='{"customer_id": 2, "tenure": 5, "contract": "Two year"}'
                      )
print("Mensagem publicada!")

Mensagem publicada!


In [3]:
def processar_sem_ack(ch, method, properties, body):
    import json
    dados = json.loads(body)
    print("Mensagem recebida:", dados)
    print("Processamento 'travou'... sem ACK!")

channel.basic_consume(queue='lab_churn', on_message_callback=processar_sem_ack)
print("Aguardando mensagens...")

Aguardando mensagens...


In [4]:
connection.process_data_events(time_limit=5)  # Processa mensagens por 5 segundos

Mensagem recebida: {'customer_id': 2, 'tenure': 5, 'contract': 'Two year'}
Processamento 'travou'... sem ACK!


In [5]:
connection.close()
print("Conexão fechada!")

Conexão fechada!


### Sessão 5 - Aplicando ao projeto de Churn

- Aplicando todo esse conceito de mensagens no nosso atual projeto de previsão de churn
podemos pensar que o primeiro producer seria a requisição vinda do CRM com os dados brutos dos clientes
e o proximo consumer será o serviço responsável por processar os dados, fazendo feature enginnering 
e posteriormente viraria o producer do proximo serviço que seria o responsável por fazer a previsao
e depois publicaria a resposta da previsao.

- Como body da mensagem inicial seria um json com dados do cliente e teria o mesmo formato de
CustomerInput. Já o de predição seria um json com estrutura de PredictionOutput.

- Esse resultado seria salvo em um banco de dados.